# CS2310 — From Control Flow to Procedural Abstraction
## A score analyser, built one decision at a time

**Original teaching notebook · C++20 · approximately 75–90 minutes.**

We will build a small score-analysis tool. Start with one mark; then classify a result, aggregate a class, retain marks in arrays, extract reusable functions and combine them. The problem changes as we introduce new language elements.

**How to work:** predict each result before reading the explanation, run the complete program separately, then explain the difference between what it does and what it should do. Normal demonstration cells are independent translation units: compile each separately with `g++ -std=c++20 example.cpp -o example`. Do not concatenate their multiple main functions. This avoids relying on persistent Jupyter state. The C++ kernel or notebook tooling may need separate setup; ClassLab imports the source without requiring a kernel.

Five marked cells are classroom tasks. Their tests and teacher notes are in Cell Metadata. The instructor should retain this file and distribute tasks through ClassLab, not share the full teacher notebook with students. Reference implementations are a separate instructor file. All data are synthetic.


## 1. A score is a value, not a string of digits
A mark has a domain: 0 through 100. An integer variable stores a value; an expression computes a value; assignment replaces the value in an existing object. Watch the distinction between declaring a variable, initialising it, and changing it.


In [ ]:
#include <iostream>

int main() {
    int raw = 78;
    int bonus = 5;
    int adjusted = raw + bonus;
    std::cout << raw << ' ' << bonus << ' ' << adjusted << '\n';
}


**Trace:** raw = 78, bonus = 5, adjusted = 83. Evaluate the right-hand side before the assignment. Changing raw later does not automatically recalculate adjusted. What does the next program print?


In [ ]:
#include <iostream>

int main() {
    int raw = 78;
    int adjusted = raw + 5;
    raw = 60;
    std::cout << raw << ' ' << adjusted << '\n';
}


The output is `60 83`. Variables are not spreadsheet formulas. This is important when a program stores the current class total: updating a score does not update a previously computed statistic.


### Input and output: a contract with the user
A batch program reads a sequence of tokens from standard input and writes answers to standard output. Separate the input data from any prompts. An automated judge normally compares output, not the text printed in an interactive terminal. A valid score is guaranteed in the following small example.


In [ ]:
#include <iostream>

int main() {
    int score;
    if (!(std::cin >> score)) return 1;
    std::cout << score + 2 << '\n';
}


Try the input `71`: output `73`. Try whitespace before the input: formatted extraction skips it. The stream test prevents reading an uninitialised variable if input fails. In later exercises the input domain is specified and valid data are guaranteed.


### Types determine operations
Two integer operands cause integer division, even if the receiving object has type double. Conversion before division changes the operation. Predict the two values below.


In [ ]:
#include <iostream>

int main() {
    int total = 161, count = 2;
    double truncated = total / count;
    double precise = static_cast<double>(total) / count;
    std::cout << truncated << ' ' << precise << '\n';
}


Output: `80 80.5`. The fractional part is lost during the integer operation, before assignment. With nonnegative scores integer division truncates down. The representation choice matters for an average; we will return to this when extracting a function.


## 2. Conditional execution: classifying one mark
We define the exercise's pass rule: scores at least 60 pass. The Boolean expression `score >= 60` has value true or false. Unlike `score = 60`, it does not change the score. Treat the boundary as part of the specification, not an arbitrary constant.


In [ ]:
#include <iostream>

int main() {
    int score = 60;
    bool passed = score >= 60;
    std::cout << std::boolalpha << passed << '\n';
}


Expected `true`. Use a decision table: 59 → FAIL; 60 → PASS; 61 → PASS. Testing only 80 and 30 cannot distinguish `>` from `>=`.


In [ ]:
#include <iostream>

int main() {
    int score = 59;
    if (score >= 60) {
        std::cout << "PASS\n";
    } else {
        std::cout << "FAIL\n";
    }
}


Only one branch executes. A sequence of independent if statements can be useful when conditions are independent; here the alternatives are mutually exclusive. Try score 60 by changing the initialiser in a scratch copy.


### Multiple categories
For this classroom example, bands are A (85–100), B (70–84), C (60–69), F (0–59). Compare from the most restrictive threshold first. The order ensures that a mark of 90 does not also get B or C.


In [ ]:
#include <iostream>

int main() {
    int score = 88;
    if (score >= 85) std::cout << "A\n";
    else if (score >= 70) std::cout << "B\n";
    else if (score >= 60) std::cout << "C\n";
    else std::cout << "F\n";
}


The output is `A`. A useful alternative is a decision table. If the lower thresholds are checked first, the higher band may become unreachable. Trace 59, 60, 69, 70, 84, 85 and 100 before changing any conditions.


### Classroom interaction 1 — Fix the grade boundary
**Learning objective:** implement an inclusive boundary. Read one integer score (0–100). Print exactly PASS for scores ≥60, otherwise FAIL, followed by a newline. The following starter compiles, but one comparison fails at the boundary. Locate the error, repair it, and use a public case to verify. A passing mark is an academic rule defined here, not the university's grading policy.


In [ ]:
%%cpp
// @classlab exercise: Fix the grade boundary
#include <iostream>

int main() {
    int score;
    std::cin >> score;
    if (score > 60) std::cout << "PASS\n";
    else std::cout << "FAIL\n";
}


**Debrief:** Why is the equality case more diagnostic than the obvious extremes? A single carefully chosen test can expose the fault. Explain which Boolean condition is true without relying on a screenshot of the output.


## 3. Repetition: collecting class totals
Instead of writing one statement for each score, keep a running total. For a sequence of values, maintain the invariant: before processing the next score, sum equals the total of all scores already processed. Start with zero because no scores have been processed.


In [ ]:
#include <iostream>

int main() {
    int sum = 0;
    for (int i = 1; i <= 4; ++i) {
        sum += 10 * i;
        std::cout << i << ": " << sum << '\n';
    }
}


The running totals are 10, 30, 60, 100. The loop has three parts: initialisation (once), condition (before each iteration), increment (after the body). With a zero-based counter for n values, the visited indices are 0 through n−1.


In [ ]:
#include <iostream>

int main() {
    int sum = 0;
    for (int i = 0; i < 3; ++i) {
        int score;
        std::cin >> score;
        sum += score;
    }
    std::cout << sum << '\n';
}


For input `10 20 30` the answer is 60. This streaming design does not require storing all marks. Use a while loop if the number of entries depends on a condition instead of a known count.


In [ ]:
#include <iostream>

int main() {
    int sum = 0;
    int score;
    while (std::cin >> score && score != -1) {
        sum += score;
    }
    std::cout << sum << '\n';
}


For `10 20 -1` the output is 30. Here −1 is a sentinel, deliberately outside the score domain. Do not add the sentinel. Using the extraction as part of the condition also prevents repeated use of stale data when the input ends.


### Counting and summing are different
Count how many marks satisfy a condition while separately summing the marks. The total number of scores is not necessarily the number of passing scores.


In [ ]:
#include <iostream>

int main() {
    int count = 0, sum = 0;
    for (int i = 0; i < 4; ++i) {
        int score;
        std::cin >> score;
        sum += score;
        if (score >= 60) ++count;
    }
    std::cout << sum << ' ' << count << '\n';
}


Input `60 80 59 1` gives `200 2`. A loop can maintain several invariants simultaneously: sum = total of processed scores, count = passing scores among those processed. Notice how boundary reasoning from Section 2 reappears in a loop.


### Classroom interaction 2 — Debug the accumulation loop
**Learning objective:** find the off-by-one error. First read n (1–10), then n integer scores (0–100). Output their sum. The starter silently skips a score. Test a class with one student as well as a larger class.


In [ ]:
// @classlab exercise: Debug the accumulation loop
#include <iostream>

int main() {
    int n;
    std::cin >> n;
    int total = 0;
    for (int i = 0; i < n - 1; ++i) {
        int score;
        std::cin >> score;
        total += score;
    }
    std::cout << total << '\n';
}


**Debrief:** the initial value zero is correct for a sum, but the loop must execute exactly n times. For input 1 followed by 73, trace the condition before the first iteration. Why does a last score of zero hide this bug?


## 4. Arrays: keeping the scores for later passes
Streaming is enough for a total, but we may want to inspect the original marks, calculate deviations from the average or revisit an earlier score. An array holds a fixed number of objects of the same type. With a length of four, valid indices are 0, 1, 2, 3, never 4.


In [ ]:
#include <iostream>

int main() {
    int scores[4] = {72, 60, 91, 55};
    std::cout << scores[0] << ' ' << scores[3] << '\n';
    scores[1] += 5;
    std::cout << scores[1] << '\n';
}


Output is `72 55` and then `65`. Indexing reads or writes the selected element. An out-of-range access has undefined behaviour; do not demonstrate it by executing an invalid access. Use the domain of valid indices as a proof obligation.


In [ ]:
#include <iostream>

int main() {
    int scores[4] = {72, 60, 91, 55};
    for (int i = 0; i < 4; ++i) {
        std::cout << i << ": " << scores[i] << '\n';
    }
}


The index is useful when you need the position. A range-based for loop is more direct if you only need the elements of an entire fixed array. For a partially filled capacity array, iterate only through the logical size n.


In [ ]:
#include <iostream>

int main() {
    int scores[10] = {};
    int n = 3;
    scores[0] = 72; scores[1] = 60; scores[2] = 91;
    int sum = 0;
    for (int i = 0; i < n; ++i) sum += scores[i];
    std::cout << sum << '\n';
}


The result 223 is calculated from the three active marks. The capacity (10) and logical size (3) are not interchangeable. C++ arrays with a runtime-variable length are not portable standard C++20; for this bounded lesson a fixed capacity suffices.


### Maximum and the initial candidate
For a nonempty sequence, choose its first element as the initial maximum. Then inspect each remaining element. Invariant: best is the maximum of the elements examined so far. Choosing zero happens to work for nonnegative values, but fails when we apply the same algorithm to score *changes* which may be negative.


In [ ]:
#include <iostream>

int main() {
    int changes[4] = {-8, -3, -12, -5};
    int best = changes[0];
    for (int i = 1; i < 4; ++i) {
        if (changes[i] > best) best = changes[i];
    }
    std::cout << best << '\n';
}


Output is −3, not 0. A candidate that is not present in the input is a dangerous default. If n=0, no maximum exists under this specification; we require n≥1.


### Classroom interaction 3 — Find the maximum correctly
**Learning objective:** choose an invariant-preserving initial value. Read n (1–10), then n integer changes in marks (−100 to 100). Print the greatest change. Negative changes represent a decrease from a prior mark. The starter assumes 0 is a safe initial maximum.


In [ ]:
// @classlab exercise: Find the maximum correctly
#include <iostream>

int main() {
    int n;
    std::cin >> n;
    int changes[10] = {};
    for (int i = 0; i < n; ++i) std::cin >> changes[i];
    int best = 0;
    for (int i = 0; i < n; ++i) {
        if (changes[i] > best) best = changes[i];
    }
    std::cout << best << '\n';
}


**Debrief:** after processing the first change, the invariant should hold even if it is negative. Starting from the first element and retaining the original loop from index 0 is correct (one redundant comparison); starting comparisons at index 1 is also correct but is a different source from the demo's exact reference.


## 5. Procedural abstraction: name a computation
Our main programs are growing. Extract a function when it has a clear contract. A function declaration communicates the return type and parameters; the definition contains the implementation; a call supplies arguments. Separate the calculation from formatted output so the calculation can be reused.


In [ ]:
#include <iostream>
bool passed(int score) {
    return score >= 60;
}
int main() {
    std::cout << std::boolalpha << passed(59) << ' ' << passed(60) << '\n';
}


Output: `false true`. The parameter score is local to the function. A call evaluates the argument, initialises the parameter and returns the result. The caller need not know the exact form of the comparison.


In [ ]:
#include <iostream>
int with_bonus(int score, int bonus) {
    score += bonus;
    return score;
}
int main() {
    int original = 75;
    int adjusted = with_bonus(original, 5);
    std::cout << original << ' ' << adjusted << '\n';
}


Output: `75 80`. Passing an int by value copies the value; modifying the local parameter does not modify the caller's variable. Passing by non-const reference makes intentional mutation explicit.


In [ ]:
#include <iostream>
void add_bonus(int& score, int bonus) {
    score += bonus;
}
int main() {
    int original = 75;
    add_bonus(original, 5);
    std::cout << original << '\n';
}


Output: `80`. A reference parameter aliases the caller's object. Prefer returning a value when the operation naturally computes a result; use a mutable reference only when changing the original is the intent.


### Passing arrays and their logical sizes
A parameter written as const int scores[] in this simple function is adjusted to a pointer. The length is not communicated by the parameter type. We pass n explicitly; const prevents this function from modifying the elements.


In [ ]:
#include <iostream>
int sum_scores(const int scores[], int n) {
    int total = 0;
    for (int i = 0; i < n; ++i) total += scores[i];
    return total;
}
int main() {
    int scores[] = {60, 80, 91};
    std::cout << sum_scores(scores, 3) << '\n';
}


The output is 231. The function's caller guarantees the first n elements exist. Avoid computing the array length with sizeof inside the function: the parameter is a pointer. We can implement average in terms of sum_scores and n, under the precondition n>0.


In [ ]:
#include <iostream>
#include <iomanip>
double average(int total, int count) {
    return static_cast<double>(total) / count;
}
int main() {
    std::cout << std::fixed << std::setprecision(2) << average(161, 2) << '\n';
}


Output: `80.50`. fixed and setprecision(2) control display, not the underlying division. A return type of double does not by itself make an integer division floating-point. A count of zero is outside the function contract.


### Classroom interaction 4 — Fix the average function
**Learning objective:** distinguish return type from expression type. Read n (1–10) followed by n integer marks (0–100), print their mean with exactly two decimal places. The starter has a reusable function but a type error in its arithmetic.


In [ ]:
// @classlab exercise: Fix the average function
#include <iostream>
#include <iomanip>

double average(int total, int count) {
    return total / count;
}

int main() {
    int n;
    std::cin >> n;
    int total = 0;
    for (int i = 0; i < n; ++i) {
        int score;
        std::cin >> score;
        total += score;
    }
    std::cout << std::fixed << std::setprecision(2) << average(total, n) << '\n';
}


**Debrief:** why does changing the return type not fix the expression? The operator is evaluated first. Tests with divisible totals cannot reveal this error. Include a non-integral mean and check formatting separately.


## 6. Combine the pieces: one score analyser
We have learned to classify, aggregate, store and abstract computations. We now design one program to produce three statistics: total, maximum, and passing count. Use a first pass to read n scores into an array, then compute statistics. Keep the input contract simple: n is 1–10, scores are integers 0–100.


### Plan before implementation
For input `4 72 60 91 55`, total = 278, maximum = 91, passing count = 3. A table of state after each processed score is more useful than starting immediately with code.

| Processed | Total | Best | Passes |
|---|---:|---:|---:|
| 72 | 72 | 72 | 1 |
| 60 | 132 | 72 | 2 |
| 91 | 223 | 91 | 3 |
| 55 | 278 | 91 | 3 |

Initialise total and passes to zero, and best to the first score. Pass the logical size into each function. Independent functions can be tested on the same array.


In [ ]:
#include <iostream>
int count_passes(const int scores[], int n) {
    int passes = 0;
    for (int i = 0; i < n; ++i)
        if (scores[i] >= 60) ++passes;
    return passes;
}
int main() {
    int marks[] = {72, 60, 91, 55};
    std::cout << count_passes(marks, 4) << '\n';
}


The result is 3. Notice the function does not read from cin or print; input and output are responsibilities of main. The same calculation can be reused for a different class or tested with a small array. We can choose separate passes for clarity; a single combined pass is also possible.


In [ ]:
#include <iostream>
int maximum(const int values[], int n) {
    int best = values[0];
    for (int i = 1; i < n; ++i)
        if (values[i] > best) best = values[i];
    return best;
}
int main() {
    int marks[] = {0, 100, 60};
    std::cout << maximum(marks, 3) << '\n';
}


Output is 100. When testing a maximum routine, vary the location of the maximum (first, middle, last) and test a singleton. When testing passing count, include the exact threshold. The integrated task also requires that all three numbers appear in the prescribed order.


### Classroom interaction 5 — Complete the score analyser
**Learning objective:** transfer earlier ideas into independent program construction. Read n (1–10) and n integer scores in [0,100]. Print three integers on one line separated by a single space: total, maximum, number of scores ≥60. Implement the three TODO functions in the starter. Keep main's input/output contract unchanged. All-zero scores are valid; n is never zero.


In [ ]:
// @classlab exercise: Complete the score analyser
#include <iostream>

int total_score(const int scores[], int n) {
    // TODO: aggregate all n scores.
    return 0;
}

int maximum(const int scores[], int n) {
    // TODO: find the maximum.
    return 0;
}

int count_passes(const int scores[], int n) {
    // TODO: count scores at least 60.
    return 0;
}

int main() {
    int n;
    std::cin >> n;
    int scores[10] = {};
    for (int i = 0; i < n; ++i) std::cin >> scores[i];
    std::cout << total_score(scores, n) << ' ' << maximum(scores, n)
              << ' ' << count_passes(scores, n) << '\n';
}


### Wrap-up: how to debug systematically
1. Specify the input domain and expected output before coding.
2. Trace one minimal example; identify an invariant.
3. Test boundary cases and examples that distinguish the wrong algorithm from the intended one.
4. Extract a function with a clear precondition and return value.
5. Run an integrated test only after the component contracts are clear.

Compare the five debugging patterns: an inclusive condition, a loop's last iteration, a maximum's initial candidate, an arithmetic expression's type, and translating requirements into function bodies.


### Instructor's ClassLab rehearsal (teacher only)
This section is marked as a teacher reference. For the one-minute demo, upload this Notebook and choose Fix the grade boundary. All five marked exercises have fixed Demo fixtures for exactly the starter and instructor reference source. Free-form Run accepts only the predefined public inputs. Other solutions may be perfectly correct yet show no fixture; real C++20 judging requires the separately validated Judge. Do not copy the teacher notebook or references into a student task.
